In [2]:
import boto3
import pandas as pd
import numpy as np

AWS_REGION = "ap-south-1"
BUCKET_NAME = "krushang-beverage-ml-2026"

RAW_KEY = "raw/survey_results.csv"
PROCESSED_PREFIX = "processed"

s3 = boto3.client("s3", region_name=AWS_REGION)

print("AWS Region:", AWS_REGION)
print("S3 Bucket:", BUCKET_NAME)

AWS Region: ap-south-1
S3 Bucket: krushang-beverage-ml-2026


In [3]:
response = s3.get_object(
    Bucket=BUCKET_NAME,
    Key=RAW_KEY
)

df = pd.read_csv(response["Body"])

print("Raw dataset shape:", df.shape)
df.head()

Raw dataset shape: (30010, 17)


,respondent_id,age,gender,zone,occupation,income_levels,consume_frequency(weekly),current_brand,preferable_consumption_size,awareness_of_other_brands,reasons_for_choosing_brands,flavor_preference,purchase_channel,packaging_preference,health_concerns,typical_consumption_situations,price_range
0,R00001,30,M,Urban,Working Professional,<10L,3-4 times,Newcomer,Medium (500 ml),0 to 1,Price,Traditional,Online,Simple,Medium (Moderately health-conscious),"Active (eg. Sports, gym)",100-150
1,R00002,46,F,Metro,Working Professional,> 35L,5-7 times,Established,Medium (500 ml),2 to 4,Quality,Exotic,Retail Store,Premium,Medium (Moderately health-conscious),Social (eg. Parties),200-250
2,R00003,41,F,Rural,Working Professional,> 35L,3-4 times,Newcomer,Medium (500 ml),2 to 4,Availability,Traditional,Retail Store,Premium,Medium (Moderately health-conscious),"Active (eg. Sports, gym)",200-250
3,R00004,33,F,Urban,Working Professional,16L - 25L,5-7 times,Newcomer,Medium (500 ml),0 to 1,Brand Reputation,Exotic,Online,Eco-Friendly,Low (Not very concerned),"Active (eg. Sports, gym)",150-200
4,R00005,23,M,Metro,Student,NaN,3-4 times,Established,Medium (500 ml),0 to 1,Availability,Traditional,Online,Premium,Medium (Moderately health-conscious),"Active (eg. Sports, gym)",50-100


In [4]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

Shape: (30010, 17)

Columns:
['respondent_id', 'age', 'gender', 'zone', 'occupation', 'income_levels', 'consume_frequency(weekly)', 'current_brand', 'preferable_consumption_size', 'awareness_of_other_brands', 'reasons_for_choosing_brands', 'flavor_preference', 'purchase_channel', 'packaging_preference', 'health_concerns', 'typical_consumption_situations', 'price_range']

Missing values:
respondent_id                        0
age                                  0
gender                               0
zone                                 0
occupation                           0
income_levels                     8064
consume_frequency(weekly)            8
current_brand                        0
preferable_consumption_size          0
awareness_of_other_brands            0
reasons_for_choosing_brands          0
flavor_preference                    0
purchase_channel                    10
packaging_preference                 0
health_concerns                      0
typical_consumption_situa

In [5]:
# Step 1: Remove exact duplicate rows

print("Shape before duplicate removal:", df.shape)
print("Exact duplicates:", df.duplicated().sum())

df = df.drop_duplicates().copy()

print("Shape after duplicate removal:", df.shape)

Shape before duplicate removal: (30010, 17)
Exact duplicates: 10
Shape after duplicate removal: (30000, 17)


In [6]:
# Step 2: Check categorical inconsistencies

for col in ["zone", "current_brand"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


--- zone ---
zone
Metro         11911
Urban         10688
Semi-Urban     5275
Rural          2117
urbna             5
Metor             4
Name: count, dtype: int64

--- current_brand ---
current_brand
Established    15447
Newcomer       14503
newcomer          30
Establishd        20
Name: count, dtype: int64


In [7]:
# Step 3: Fix confirmed categorical inconsistencies

df["zone"] = df["zone"].replace({
    "urbna": "Urban",
    "Metor": "Metro"
})

df["current_brand"] = df["current_brand"].replace({
    "newcomer": "Newcomer",
    "Establishd": "Established"
})

In [9]:
print(df["zone"].value_counts(dropna=False))
print()
print(df["current_brand"].value_counts(dropna=False))

zone
Metro         11915
Urban         10693
Semi-Urban     5275
Rural          2117
Name: count, dtype: int64

current_brand
Established    15467
Newcomer       14533
Name: count, dtype: int64
